# Add SageMaker training and S3 integration



## Startup cells

In [2]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '5y1nkdhiat07fr'
os.environ['DataZoneDomainId'] = 'dzd-baa51un8enpy07'
os.environ['DataZoneEnvironmentId'] = '6g07zm1q2jqsyv'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "5y1nkdhiat07fr",
                "DataZoneDomainId": "dzd-baa51un8enpy07",
                "DataZoneEnvironmentId": "6g07zm1q2jqsyv",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [3]:
import boto3
s3 = boto3.client("s3")
response = s3.list_buckets()

[b["Name"] for b in response["Buckets"]]

['amazon-sagemaker-153058521958-us-east-1-5y1nkdhiat07fr',
 'cloud-ml-genai-api-proj-153058521958-us-east-1-an']

In [5]:
import pandas as pd

bucket = "cloud-ml-genai-api-proj-153058521958-us-east-1-an"

df = pd.read_csv( f"s3://{bucket}/data/raw/creditcard.csv")

df.head()

ImportError: `Import fsspec` failed.  Use pip or conda to install the fsspec package.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import mlflow
import mlflow.sklearn

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

mlflow.set_experiment("fraud_detection_sagemaker")

with mlflow.start_run():

    model = RandomForestClassifier(n_estimators=100, random_state=42)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    report = classification_report(y_test, preds, output_dict=True)

    mlflow.log_metric(
        "f1_score", report["1"]["f1-score"]
    )

    mlflow.sklearn.log_model( model, "fraud_model")

    print(report)

ModuleNotFoundError: No module named 'sklearn'

In [0]:
import os
import mlflow

bucket = "cloud-ml-genai-api-proj-153058521958-us-east-1-an"

mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("fraud_detection_sagemaker")

print("Tracking URI:", mlflow.get_tracking_uri())

/sagemaker_packages/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/11 15:37:39 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_sagemaker' does not exist. Creating a new experiment.


Tracking URI: file:///tmp/mlruns


In [0]:
import joblib
import boto3

model_path = "/tmp/fraud_model.pkl"
joblib.dump(model, model_path)

s3 = boto3.client("s3")

s3.upload_file(model_path, bucket, "models/fraud_model.pkl")

print(f"Uploaded model to s3://{bucket}/models/fraud_model.pkl")

Uploaded model to s3://cloud-ml-genai-api-proj-153058521958-us-east-1-an/models/fraud_model.pkl


In [7]:
# Validate the saved model

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="models/"
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

models/ 0
models/fraud_model.pkl 2615641


In [0]:
# syncing local Mlruns to s3 

!aws s3 sync ./mlruns s3://cloud-ml-genai-api-proj-153058521958-us-east-1-an/mlflow/

In [16]:
# downloading the model from s3 to localimport boto3

# Initialize the S3 client
s3 = boto3.client('s3')

# Define your variables with quotation marks
s3_key = 'models/fraud_model.pkl'
local_path = 'fraud_model.pkl'

# Download the file
s3.download_file(bucket, s3_key, local_path)


## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()